In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import pickle
import utils
import algo
from scipy.stats import pearsonr
%matplotlib widget

## Prepare data

In [ ]:
def prepare_data_subj(Subj_ID, fs):
    eeg_list, eog_list, gaze_list, feats_list = utils.load_subj(Subj_ID)
    gaze_coords_list = [gaze[:,0:2,:] for gaze in gaze_list]
    saccade_list = [np.expand_dims(gaze[:,2,:], axis=1) for gaze in gaze_list]
    blink_list = [np.expand_dims(gaze[:,3,:], axis=1) for gaze in gaze_list]
    saccade_list = utils.refine_saccades(saccade_list, blink_list)
    gaze_velocity_list = [utils.calcu_gaze_velocity(gaze) for gaze in gaze_list]
    objflow_list = [np.expand_dims(feats[:,8,:], axis=1) for feats in feats_list]
    eeg_reg_list = [utils.regress_out(eeg, eog) for eeg, eog in zip(eeg_list, eog_list)]
    eeg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eeg_list]
    eog_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eog_list]
    gaze_coords_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in gaze_coords_list]
    saccade_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1, CENTER=False) for d in saccade_list]
    blink_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1, CENTER=False) for d in blink_list]
    gaze_velocity_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in gaze_velocity_list]
    objflow_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in objflow_list]
    eeg_reg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=1) for d in eeg_reg_list]
    data_multitask_dict = {'EEG': eeg_list, 'EOG': eog_list, 'GAZE': gaze_coords_list, 'GAZE_V': gaze_velocity_list, 'EEG-EOG': eeg_reg_list}
    return data_multitask_dict, objflow_list, saccade_list, blink_list

def find_most_correlated_segment(series_a, series_b):
    len_a = len(series_a)
    len_b = len(series_b)
    if len_b > len_a:
        raise ValueError("Series B cannot be longer than series A")
    # Calculate correlations for all possible segments
    correlations = []
    for i in range(len_a - len_b + 1):
        segment = series_a[i:i + len_b]
        corr, p_value = pearsonr(segment, series_b)

        correlations.append({
            'correlation': corr,
            'p_value': p_value,
            'start_idx': i,
            'end_idx': i + len_b,
            'segment': segment.copy()
        })
    # Find the segment with highest absolute correlation
    best_match = max(correlations, key=lambda x: abs(x['correlation']))
    return {
        'best_correlation': best_match['correlation'],
        'p_value': best_match['p_value'],
        'start_index': best_match['start_idx'],
        'end_index': best_match['end_idx'],
        'best_segment': best_match['segment'],
        'all_correlations': [c['correlation'] for c in correlations]
    }

def plot_correlation_results(series_a, series_b, result):
    """
    Plot the original series and highlight the best matching segment.
    """
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 10))
    
    # Plot full series A with highlighted segment
    ax1.plot(series_a, label='Series A', alpha=0.7)
    start_idx = result['start_index']
    end_idx = result['end_index']
    ax1.plot(range(start_idx, end_idx), result['best_segment'], 
             color='red', linewidth=2, label=f'Best segment (r={result["best_correlation"]:.3f})')
    ax1.set_title('Series A with Best Matching Segment Highlighted')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot correlation along the sliding window
    ax2.plot(result['all_correlations'])
    ax2.axhline(y=result['best_correlation'], color='red', linestyle='--', 
                label=f'Best correlation: {result["best_correlation"]:.3f}')
    ax2.axvline(x=start_idx, color='red', linestyle='--', alpha=0.5)
    ax2.set_title('Correlation Along Sliding Window')
    ax2.set_xlabel('Starting Position')
    ax2.set_ylabel('Correlation')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Compare best segment with series B
    ax3.plot(result['best_segment'], label='Best segment from A', marker='o', markersize=3)
    ax3.plot(series_b, label='Series B', marker='s', markersize=3)
    ax3.set_title(f'Comparison: Best Segment vs Series B (r={result["best_correlation"]:.3f})')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
subjects = ['SUB_01', 'SUB_02', 'SUB_03', 'SUB_04', 'SUB_05', 'SUB_06', 'SUB_07', 'SUB_08', 'SUB_09', 'SUB_10', 'SUB_11', 'SUB_12', 'SUB_13', 'SUB_14', 'SUB_15', 'SUB_16', 'SUB_17', 'SUB_18', 'SUB_19']
# load the data of the first protocol
fs = 30
with open('data/1stprotocol/features.pkl', 'rb') as f:
    features_list = pickle.load(f)
with open(f"data/1stprotocol/eeg_multisub_noreg.pkl", 'rb') as f:
    eeg_multisub_list = pickle.load(f)
with open('data/1stprotocol/eog_multisub.pkl', 'rb') as f:
    eog_multisub_list = pickle.load(f)

In [ ]:
video_idx = [2, 3, 5, 6, 8, 10, 12]
eeg_list = [eeg_multisub_list[vid] for vid in video_idx]
eog_list = [eog_multisub_list[vid] for vid in video_idx]
objflow_list = [features_list[vid][:,8] for vid in video_idx]

In [ ]:
_, objflow_ref_list, _, _ = prepare_data_subj(0, fs)

In [ ]:
for i in range(len(video_idx)):
    result = find_most_correlated_segment(objflow_list[i], objflow_ref_list[i][:,0,0])
    eeg_list[i] = eeg_list[i][result['start_index']:result['end_index'], ...]
    eog_list[i] = eog_list[i][result['start_index']:result['end_index'], ...]
    objflow_list[i] = objflow_ref_list[i][:,0,0]


In [ ]:
eeg_reg_list = [utils.regress_out(eeg, eog) for eeg, eog in zip(eeg_list, eog_list)]
# just centering
objflow_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=0) for d in objflow_list]
eeg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=0) for d in eeg_list]
eeg_reg_list = [utils.remove_shot_cuts_and_center(d, fs, remove_time=0) for d in eeg_reg_list]

In [ ]:
eeg_list = [eeg[:,:,:, np.newaxis] for eeg in eeg_list]
eeg_reg_list = [eeg_reg[:,:,:, np.newaxis] for eeg_reg in eeg_reg_list]
objflow_list = [objflow[:, np.newaxis, np.newaxis] for objflow in objflow_list]

In [ ]:
EEG_REG = True

## EEG-Stim Analyze

In [ ]:
L_EEG = 3
L_Stim = 15
offset_EEG = 1
offset_Stim = 0

RUN_TIMES = 1
MOD = 'EEG-EOG' if EEG_REG else 'EEG'
trial_len = 45
task_train = [1]
nb_nearby_samples = None # [9, 3]
BOOTSTRAP = False
n_components = 5 if (MOD != 'GAZE_V' and MOD != 'GAZE') else 3
range_into_account = 3
nb_comp_into_account = 2

In [ ]:
def analyze_all(eeg_reg_list, objflow_list, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, range_into_account, nb_comp_into_account, task_train=[1], trial_len=30, n_components=5, save_name=None, MOD='EEG-EOG', PERMU_TEST=False, BOOTSTRAP=True, nb_nearby_samples=None, nb_mismatch=None):
    all_acc = []
    corr_match_all_subj = []
    corr_mismatch_all_subj = []
    acc_permu_all = []
    start_points = None
    for Subj_ID in range(len(subjects)):
        print(f"###################\nSubject {Subj_ID + 1} / {len(subjects)}")
        data_list = [eeg_reg[:,:,Subj_ID,:] for eeg_reg in eeg_reg_list]
        data_masked_list = None
        objflow_masked_list = None
        CCA = algo.CanonicalCorrelationAnalysis(data_list, objflow_list, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, task_train=task_train, leave_out=1, n_components=n_components, EEG_masked=data_masked_list, Stim_masked=objflow_masked_list)
        nb_compete = nb_mismatch if nb_mismatch is not None else 1
        corr_match_data, corr_mismatch_data, acc_permu_list, start_points, _ = CCA.match_mismatch(trial_len=trial_len, PERMU_TEST=PERMU_TEST, BOOTSTRAP=BOOTSTRAP, given_start_points=start_points, nb_compete=nb_compete)
        print("###########Match-Mismatch, TASK 1, 2, 3###########")
        acc_all_tasks, _, _, _, _ = utils.eval_compete_3D(corr_match_data, corr_mismatch_data, True, range_into_account=range_into_account, nb_comp_into_account=nb_comp_into_account, message=True)
        all_acc.append({
            'Subject': Subj_ID + 1,
            'Task_1': acc_all_tasks[0],
        })
        corr_match_all_subj.append(corr_match_data)
        corr_mismatch_all_subj.append(corr_mismatch_data)
        if PERMU_TEST:
            acc_permu_all += acc_permu_list
    all_acc = pd.DataFrame(all_acc)
    if PERMU_TEST:
        acc_permu = np.concatenate(acc_permu_all, axis=0)
        alpha = 0.05
        lower_bound = np.percentile(acc_permu, alpha/2*100)
        upper_bound = np.percentile(acc_permu, (1-alpha/2)*100)
    else:
        lower_bound = None
        upper_bound = None
    # add two columns to all_acc for lower and upper bound
    all_acc['lower_bound'] = lower_bound
    all_acc['upper_bound'] = upper_bound
    if save_name is not None:
        save_path = f"tables/{MOD}/{save_name}"
        all_acc.to_csv(f"{save_path}_acc_{trial_len}_train_{task_train}{'_BT' if BOOTSTRAP else ''}{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}{('_nbmm'+str(nb_mismatch)) if nb_mismatch is not None else ''}_1stprotocol.csv", index=False)
        # save corr_match_all_subj, corr_mismatch_all_subj, start_idx as dictionary
        corr_res = {
            'corr_match_all_subj': corr_match_all_subj,
            'corr_mismatch_all_subj': corr_mismatch_all_subj,
            'start_points': start_points
        }
        # save as pickle file
        with open(f"{save_path}_corr_{trial_len}_train_{task_train}{'_BT' if BOOTSTRAP else ''}{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}{('_nbmm'+str(nb_mismatch)) if nb_mismatch is not None else ''}_1stprotocol.pickle", 'wb') as f:
            pickle.dump(corr_res, f)
    return all_acc, corr_match_all_subj, corr_mismatch_all_subj, start_points

In [ ]:
# create a dictionary to store the results
for i in range(RUN_TIMES):
    save_name = f"RUN_{i+1}"
    # PERMU_TEST = (i == 0)
    PERMU_TEST = False
    nb_mismatch = None
    if EEG_REG:
        all_acc, corr_match_all_subj, corr_mismatch_all_subj, start_idx = analyze_all(eeg_reg_list, objflow_list, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, range_into_account, nb_comp_into_account, task_train=[1], trial_len=trial_len, n_components=n_components, save_name=save_name, MOD='EEG-EOG', PERMU_TEST=PERMU_TEST, BOOTSTRAP=BOOTSTRAP, nb_nearby_samples=nb_nearby_samples, nb_mismatch=nb_mismatch)
    else:
        all_acc, corr_match_all_subj, corr_mismatch_all_subj, start_idx = analyze_all(eeg_list, objflow_list, fs, L_EEG, L_Stim, offset_EEG, offset_Stim, range_into_account, nb_comp_into_account, task_train=[1], trial_len=trial_len, n_components=n_components, save_name=save_name, MOD='EEG', PERMU_TEST=PERMU_TEST, BOOTSTRAP=BOOTSTRAP, nb_nearby_samples=nb_nearby_samples, nb_mismatch=nb_mismatch)
    print(all_acc)

In [ ]:
# load results
acc_1stprotocol = []
acc_1stprotocol_noreg = []
acc_currentprotocol = []
corr_m_1stprotocol = []
corr_m_1stprotocol_noreg = []
corr_m_currentprotocol = []
for i in range(RUN_TIMES):
    save_name = f"RUN_{i+1}"
    save_path = f"tables/EEG-EOG/{save_name}"
    save_path_noreg = f"tables/EEG/{save_name}"
    acc_1stprotocol.append(pd.read_csv(f"{save_path}_acc_{trial_len}_train_[1]{'_BT' if BOOTSTRAP else ''}{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}_1stprotocol.csv"))
    acc_1stprotocol_noreg.append(pd.read_csv(f"{save_path_noreg}_acc_{trial_len}_train_[1]{'_BT' if BOOTSTRAP else ''}{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}_1stprotocol.csv"))
    acc_currentprotocol.append(pd.read_csv(f"{save_path}_acc_{trial_len}_train_[3]{'_BT' if BOOTSTRAP else ''}_masked{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}.csv"))
    with open(f"{save_path}_corr_{trial_len}_train_[1]{'_BT' if BOOTSTRAP else ''}{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}_1stprotocol.pickle", 'rb') as f:
        corr_res_1stprotocol = pickle.load(f)
        corr_m_1stprotocol.extend(corr_res_1stprotocol['corr_match_all_subj'])
    with open(f"{save_path_noreg}_corr_{trial_len}_train_[1]{'_BT' if BOOTSTRAP else ''}{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}_1stprotocol.pickle", 'rb') as f:
        corr_res_1stprotocol_noreg = pickle.load(f)
        corr_m_1stprotocol_noreg.extend(corr_res_1stprotocol_noreg['corr_match_all_subj'])
    with open(f"{save_path}_corr_{trial_len}_train_[3]{'_BT' if BOOTSTRAP else ''}_masked{('_nearby'+str(nb_nearby_samples)) if nb_nearby_samples is not None else ''}.pickle", 'rb') as f:
        corr_res_currentprotocol = pickle.load(f)
        corr_m_currentprotocol.extend(corr_res_currentprotocol['corr_match_all_subj'])
# average the results
acc_1stprotocol = pd.concat(acc_1stprotocol, ignore_index=True)
acc_1stprotocol_noreg = pd.concat(acc_1stprotocol_noreg, ignore_index=True)
acc_currentprotocol = pd.concat(acc_currentprotocol, ignore_index=True)
acc_1stprotocol = acc_1stprotocol.groupby(['Subject']).mean().reset_index()
acc_1stprotocol_noreg = acc_1stprotocol_noreg.groupby(['Subject']).mean().reset_index()
acc_currentprotocol = acc_currentprotocol.groupby(['Subject']).mean().reset_index()
corr_m_1st_all = np.concatenate(corr_m_1stprotocol, axis=0)
corr_m_1st_sum = np.max(corr_m_1st_all[:,:,0], axis=1)
corr_m_1st_noreg_all = np.concatenate(corr_m_1stprotocol_noreg, axis=0)
corr_m_1st_noreg_sum = np.max(corr_m_1st_noreg_all[:,:,0], axis=1)
corr_m_current_all = np.concatenate(corr_m_currentprotocol, axis=0)
corr_m_current_sum = np.max(corr_m_current_all[:,:,2], axis=1)

In [ ]:
# Plot the distribution of corr_m_1st_sum and corr_m_current_sum
# subsample 1000 samples from each if there are more than 1000 samples
threshold = 500
if corr_m_1st_sum.shape[0] > threshold:
    corr_m_1st_sum_red = np.random.choice(corr_m_1st_sum, threshold, replace=False)
else:
    corr_m_1st_sum_red = corr_m_1st_sum
if corr_m_1st_noreg_sum.shape[0] > threshold:
    corr_m_1st_noreg_sum_red = np.random.choice(corr_m_1st_noreg_sum, threshold, replace=False)
else:
    corr_m_1st_noreg_sum_red = corr_m_1st_noreg_sum
if corr_m_current_sum.shape[0] > threshold:
    corr_m_current_sum_red = np.random.choice(corr_m_current_sum, threshold, replace=False)
else:
    corr_m_current_sum_red = corr_m_current_sum

plt.close('all')
plt.figure(figsize=(7, 2.5))
sns.kdeplot(corr_m_1st_sum_red, label='Protocol in [8]', fill=True, alpha=0.5)
sns.kdeplot(corr_m_1st_noreg_sum_red, label='Protocol in [8] (w/o EOG regression)', fill=True, alpha=0.5)
sns.kdeplot(corr_m_current_sum_red, label='Task 3', fill=True, alpha=0.5)
# mark the 95th percentile
# plt.axvline(np.percentile(corr_m_1st_sum, 97), color='blue', linestyle='--', label='97th Percentile 1st Protocol')
# plt.axvline(np.percentile(corr_m_current_sum, 97), color='orange', linestyle='--', label='97th Percentile Current Protocol')
plt.xlabel('Correlation')
plt.ylabel('Density')
plt.xlim(-0.1, 0.4)
plt.legend()
plt.tight_layout()
plt.savefig('../../Manuscript/Spatial_Bias/corr_dist.pdf', dpi=600, bbox_inches='tight', pad_inches=0.01)
plt.show()

## Group Analysis

In [ ]:
L_gcca = 5
offset_gcca = 2

task_train = [1]
n_components = 5 if (MOD != 'GAZE_V' and MOD != 'GAZE') else 3
range_into_account = 3
nb_comp_into_account = 2

In [ ]:
SEEDs = [2, 4, 8, 16, 32]
ISC_seeds = []
for SEED in SEEDs:
    rng = np.random.default_rng(SEED)
    id_set = rng.choice(range(len(subjects)), size=14, replace=False)
    if EEG_REG:
        eeg_reg_subset = [eeg_reg[:,:,id_set,:] for eeg_reg in eeg_reg_list]
        GCCA = algo.GeneralizedCCA(eeg_reg_subset, fs, L_gcca, offset_gcca, task_train=task_train, leave_out=1, n_components=5, regularization='lwcov', message=True, signifi_level=False)
        corr_test_folds, start_points, F = GCCA.cross_val_trials(BOOTSTRAP=True, trial_len=45, given_start_points=None, BTfactor=2, CORRCA=True)
    else:
        eeg_subset = [eeg[:,:,id_set,:] for eeg in eeg_list]
        GCCA = algo.GeneralizedCCA(eeg_subset, fs, L_gcca, offset_gcca, task_train=task_train, leave_out=1, n_components=5, regularization='lwcov', message=True, signifi_level=False)
        corr_test_folds, start_points, F = GCCA.cross_val_trials(BOOTSTRAP=True, trial_len=45, given_start_points=None, BTfactor=2, CORRCA=True)
    # save corr_test_folds as pickle files
    with open(f"tables/{MOD}/isc_folds_seed_{SEED}_1stprotocol.pkl", 'wb') as f:
        pickle.dump(corr_test_folds, f)
    isc = np.mean(np.mean(corr_test_folds, axis=0).squeeze(), axis=0)
    ISC_seeds.append(isc)

In [ ]:
isc_all_1stprotocol = []
isc_all_1stprotocol_noreg = []
for SEED in SEEDs:
    with open(f"tables/EEG-EOG/isc_folds_seed_{SEED}_1stprotocol.pkl", 'rb') as f:
        corr_test_folds = pickle.load(f)
    for corr in corr_test_folds:
        isc_all_1stprotocol.append(corr)
    with open(f"tables/EEG/isc_folds_seed_{SEED}_1stprotocol.pkl", 'rb') as f:
        corr_test_folds = pickle.load(f)
    for corr in corr_test_folds:
        isc_all_1stprotocol_noreg.append(corr)
isc_all_1stprotocol = np.concatenate(isc_all_1stprotocol, axis=0)
isc_all_1stprotocol_noreg = np.concatenate(isc_all_1stprotocol_noreg, axis=0)

isc_all_task3 = []
with open(f"tables\EEG-EOG\isc_folds_train_[1, 2, 3]_eventcircle_BT.pkl", 'rb') as f:
    corr_test_folds = pickle.load(f)
for corr in corr_test_folds:
    isc_all_task3.append(corr)
isc_all_task3 = np.concatenate(isc_all_task3, axis=0)

In [ ]:
isc_1stprotocol = np.sum(isc_all_1stprotocol[:,:2,0], axis=1)
isc_1stprotocol_noreg = np.sum(isc_all_1stprotocol_noreg[:,:2,0], axis=1)
isc_currentprotocol = np.sum(isc_all_task3[:,:2,0], axis=1)
# subsample 1000 samples from each if there are more than 1000 samples
threshold = 1000
if isc_1stprotocol.shape[0] > threshold:
    isc_1stprotocol_red = np.random.choice(isc_1stprotocol, threshold, replace=False)
else:
    isc_1stprotocol_red = isc_1stprotocol
if isc_1stprotocol_noreg.shape[0] > threshold:
    isc_1stprotocol_noreg_red = np.random.choice(isc_1stprotocol_noreg, threshold, replace=False)
else:
    isc_1stprotocol_noreg_red = isc_1stprotocol_noreg
if isc_currentprotocol.shape[0] > threshold:
    isc_currentprotocol_red = np.random.choice(isc_currentprotocol, threshold, replace=False)
else:
    isc_currentprotocol_red = isc_currentprotocol

In [ ]:
# plot the pdf of isc_all
plt.figure(figsize=(7, 2.5))
sns.kdeplot(isc_1stprotocol_red, fill=True, alpha=0.5, label='Protocol in [8]')
sns.kdeplot(isc_1stprotocol_noreg_red, fill=True, alpha=0.5, label='Protocol in [8] (w/o EOG regression)')
sns.kdeplot(isc_currentprotocol_red, fill=True, alpha=0.5, label='Task 3')
plt.xlabel('ISC')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.savefig('../../Manuscript/Spatial_Bias/isc_dist.pdf', dpi=600, bbox_inches='tight', pad_inches=0.01)
plt.show()

## Visualize

In [ ]:
plt.close('all')
# Prepare data in long format for seaborn
# First protocol data
df1_long = pd.DataFrame({
    'Accuracy': acc_1stprotocol['Task_1'].values,
    'Protocol': ['Protocol in [8]'] * len(acc_1stprotocol),
    'n_subjects': [f'n={len(acc_1stprotocol)}'] * len(acc_1stprotocol)
})
lower_bound_1st = acc_1stprotocol['lower_bound'].mean()
upper_bound_1st = acc_1stprotocol['upper_bound'].mean()

# Current protocol data  
df2_long = pd.DataFrame({
    'Accuracy': acc_currentprotocol['Task_3'].values,
    'Protocol': ['Current Protocol'] * len(acc_currentprotocol),
    'n_subjects': [f'n={len(acc_currentprotocol)}'] * len(acc_currentprotocol)
})
lower_bound_current = acc_currentprotocol['lower_bound'].mean()
upper_bound_current= acc_currentprotocol['upper_bound'].mean()

# Print summary statistics
print("=== Comparison Summary ===")
print(f"1st Protocol (n={len(acc_1stprotocol)}):")
print(f"  Mean: {acc_1stprotocol['Task_1'].mean():.4f}")
print(f"  Std:  {acc_1stprotocol['Task_1'].std():.4f}")
print(f"  Min:  {acc_1stprotocol['Task_1'].min():.4f}")
print(f"  Max:  {acc_1stprotocol['Task_1'].max():.4f}")

print(f"\nCurrent Protocol (n={len(acc_currentprotocol)}):")
print(f"  Mean: {acc_currentprotocol['Task_3'].mean():.4f}")
print(f"  Std:  {acc_currentprotocol['Task_3'].std():.4f}")
print(f"  Min:  {acc_currentprotocol['Task_3'].min():.4f}")
print(f"  Max:  {acc_currentprotocol['Task_3'].max():.4f}")

# Statistical test
from scipy import stats
U_stat, p_value = stats.mannwhitneyu(acc_1stprotocol['Task_1'], acc_currentprotocol['Task_3'], alternative="greater")
print(f"\nMann-Whitney U test:")
print(f"  U-statistic: {U_stat:.4f}")
print(f"  p-value: {p_value:.4f}")

stats_comp = utils.mwu_effect(acc_1stprotocol['Task_1'], acc_currentprotocol['Task_3'], alternative="greater")
print(f"  All stats: {stats_comp}")


plt.close('all')

# Combine both datasets into one long-format dataframe
df_long = pd.concat([df1_long, df2_long], ignore_index=True)

# Overall horizontal reference lines
upper_line_1st = upper_bound_1st
upper_line_current = upper_bound_current

# Create one figure and one axis
fig, ax = plt.subplots(figsize=(3, 3))

# Boxplot (outlined only)
sns.boxplot(
    data=df_long,
    x='Protocol',
    y='Accuracy',
    color='black',
    width=0.5,
    fill=False,
    ax=ax
)

# Swarmplot (points)
sns.swarmplot(
    data=df_long,
    x='Protocol',
    y='Accuracy',
    ax=ax,
    color='black',
    alpha=0.7,
    size=4
)

# Draw per-protocol upper bound reference lines
xticks = ax.get_xticks()
labels = [t.get_text() for t in ax.get_xticklabels()]
# Map label to center x position
label_pos = dict(zip(labels, xticks))
# Small span around each category center
span = 0.35
if 'Protocol in [8]' in label_pos:
    x = label_pos['Protocol in [8]']
    ax.hlines(y=upper_line_1st, xmin=x - span, xmax=x + span, colors='grey', linestyles='--', alpha=0.8, label='Upper bound [8]')
if 'Current Protocol' in label_pos:
    x = label_pos['Current Protocol']
    ax.hlines(y=upper_line_current, xmin=x - span, xmax=x + span, colors='grey', linestyles='--', alpha=0.8, label='Upper bound Current')

# Axis styling
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_xlabel("")
ax.set_xticklabels([f"Protocol in [8]\n(n={len(acc_1stprotocol)})",
                    f"Task 3\n(n={len(acc_currentprotocol)})"])

sns.despine()

y_max = df_long['Accuracy'].max()
y_range = df_long['Accuracy'].max() - df_long['Accuracy'].min()
# Significance bracket and star for Task_1 vs Task_2
if p_value < 0.05:
    bracket_height_12 = y_max + 0.05 * y_range
    plt.plot([0, 0, 1, 1], [bracket_height_12 - 0.01 * y_range, bracket_height_12, 
                            bracket_height_12, bracket_height_12 - 0.01 * y_range], 
                'k-', linewidth=1)
    plt.text(0.5, bracket_height_12 + 0.01 * y_range, '*', 
                ha='center', va='bottom')


plt.tight_layout()
plt.show()
plt.savefig('../../Manuscript/Spatial_Bias/acc_protocols.pdf', dpi=600, bbox_inches='tight', pad_inches=0.01)



In [ ]:
isc_all_1stprotocol = []
SEEDs = [2, 4, 8, 16, 32]
for SEED in SEEDs:
    isc_all_seed = []
    with open(f"tables/EEG-EOG/isc_folds_seed_{SEED}_1stprotocol.pkl", 'rb') as f:
        corr_test_folds = pickle.load(f)
    for corr in corr_test_folds:
        isc_all_seed.append(corr)
    isc_all_1stprotocol.append(np.mean(isc_all_seed, axis=0)[:,:,0])
isc_all_1stprotocol = np.concatenate(isc_all_1stprotocol, axis=0)

isc_all = []
with open(f"tables\EEG-EOG\isc_folds_train_[1, 2, 3]_eventall_BT.pkl", 'rb') as f:
    corr_test_folds = pickle.load(f)
for corr in corr_test_folds:
    isc_all.append(corr)
isc_all_task1 = np.concatenate(isc_all, axis=0)[:,:,0]
isc_all_task2 = np.concatenate(isc_all, axis=0)[:,:,1]
isc_all_task3 = np.concatenate(isc_all, axis=0)[:,:,2]

Task_1 = np.mean(np.sum(isc_all_task1[:,:2], axis=1))
Task_2 = np.mean(np.sum(isc_all_task2[:,:2], axis=1))
Task_3 = np.mean(np.sum(isc_all_task3[:,:2], axis=1))
Ref_mean = np.mean(np.sum(isc_all_1stprotocol[:,:2], axis=1))
Ref_std = np.std(np.sum(isc_all_1stprotocol[:,:2], axis=1))
significance_level = 0.0121 # 0.0139


plt.figure(figsize=(7, 2))

# Data
task_values = [Task_1, Task_2, Task_3]
task_colors = ['blue', 'orange', 'green']
task_labels = ['Task 1 (ignore, eccentric)', 'Task 2 (attend, eccentric)', 'Task 3 (attend, central)']

# Plot task dots (circles)
for value, color, label in zip(task_values, task_colors, task_labels):
    plt.scatter(value, 1, s=60, color=color, alpha=0.8, 
               edgecolors='black', linewidth=1, label=label, marker='o')

# Plot the mean and std of ref values
plt.scatter(Ref_mean, 1, s=60, color='purple', alpha=0.8, 
            edgecolors='black', linewidth=1, label='Protocol in [8]', marker='s')
# shaded area between mean-std, mean+std
plt.fill_betweenx([0.9, 1.1], Ref_mean - Ref_std, Ref_mean + Ref_std, color='purple', alpha=0.2)

# # Plot ref dots (triangles) - only add label to first one
# for i, value in enumerate(ref_values):
#     if i == 0:  # Only add label to first reference point
#         plt.scatter(value, 1, s=100, color='orange', alpha=0.8, 
#                    edgecolors='black', linewidth=1, label='Protocol in [9]', marker='s')
#     else:  # No label for other reference points
#         plt.scatter(value, 1, s=100, color='orange', alpha=0.8, 
#                  edgecors='black', linewidth=1, marker='s')

# Significance line
plt.axvline(x=significance_level, color='grey', linestyle='--', 
           alpha=0.8, label='Significance Level')

# Clean styling
plt.xlabel('ISC (CC1 + CC2)')
plt.ylim(0.5, 1.5)
plt.yticks([])
plt.grid(True, alpha=0.3, axis='x')

# Add horizontal legend above the plot
plt.legend(bbox_to_anchor=(0.5, 1.3), loc='center', ncol=3, fontsize=10)

# Remove ot borders
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)
plt.tight_layout()
plt.show()
# plt.savefig('../../Manuscript/Spatial_Bias/isc.pdf', dpi=600, bbox_inches='tight', pad_inches=0.01)